## Cell 1 — Install dependencies

In [ ]:
!pip install -q gatspy astropy astroquery scipy numpy pandas matplotlib tqdm

## Cell 2 — Clone repo

In [ ]:
import os, sys

REPO = "/content/asteroid-pipeline"
if not os.path.exists(REPO):
    os.system(f"git clone https://github.com/wonrobot/asteroid-pipeline.git {REPO}")
else:
    os.system(f"cd {REPO} && git pull")

sys.path.insert(0, f"{REPO}/src")
print("Repo ready.")

## Cell 3 — Load RFL dataset from Google Drive

**Before running this cell**, run the following query in BigQuery and export results to Drive:

```sql
SELECT provid, obstime, band, mag, rmsmag
FROM `lsst-484623.atlast_photometry.public_obs_x05`
WHERE obstime >= '2025-04-21'
  AND obstime <= '2025-05-06'
  AND provid IS NOT NULL
ORDER BY provid, obstime
```

Export → Google Drive. Then set `DRIVE_FOLDER` below to the export folder name.

In [ ]:
from google.colab import drive
import glob, pandas as pd

drive.mount('/content/drive')

# ── Set this to your BQ export folder name in Drive ──────────────────────────
# BigQuery exports to a folder like: bq-results-YYYYMMDD-HHMMSS-xxxx
# Check your Drive and paste the folder name here.
DRIVE_FOLDER_NAME = "bq-results-20260321-232234-1774135371078"   # <-- update if needed
# ─────────────────────────────────────────────────────────────────────────────

folder = f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}"
csv_files = sorted(glob.glob(f"{folder}/*.csv"))

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in {folder}\n"
        f"Check the folder name and make sure Drive is mounted."
    )

print(f"Found {len(csv_files)} file(s): {[os.path.basename(f) for f in csv_files]}")

dfs = [pd.read_csv(f) for f in csv_files]
df_raw = pd.concat(dfs, ignore_index=True)

print(f"Loaded {len(df_raw):,} rows")
print(f"Columns: {list(df_raw.columns)}")
print(f"Sample provids: {sorted(df_raw['provid'].dropna().unique())[:8]}")
print(f"Bands: {sorted(df_raw['band'].dropna().unique())}")

# ── Results go to a subfolder of the SAME Drive folder as the input ─────────
# Survives Colab disconnects — files land in Drive, not ephemeral /content/.
RESULTS_DIR = f"{folder}/pipeline_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"\nResults will be saved to: {RESULTS_DIR}")


## Cell 4 — Configure and apply quality cuts

In [ ]:
import numpy as np
from config import PipelineConfig, OutputConfig

config = PipelineConfig(
    output=OutputConfig(
        results_dir=RESULTS_DIR,
        catalog_file=f"{RESULTS_DIR}/validation_catalog.csv",
        log_file=f"{RESULTS_DIR}/validation.log",
        verbose=True,
    ),
)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Compute MJD if not already present ───────────────────────────────────────
if "mjd" not in df_raw.columns:
    time_col = next(
        (c for c in df_raw.columns
         if c.lower() in ["obstime","datetime","time","date","obs_time","mjd_obs"]),
        None
    )
    if time_col is None:
        raise ValueError(f"No time column. Columns: {list(df_raw.columns)}")
    print(f"Computing MJD from: '{time_col}'")
    epoch = pd.Timestamp("1858-11-17")
    df_raw["mjd"] = (
        pd.to_datetime(df_raw[time_col], utc=True, errors="coerce")
        .dt.tz_localize(None).subtract(epoch)
        .dt.total_seconds().div(86400.0)
    )

print(f"Raw bands in file: {sorted(df_raw['band'].dropna().unique())}")

# ── Apply band remap BEFORE quality cuts ─────────────────────────────────────
# BigQuery returns raw band names (g/r/i). The pipeline uses canonical names
# (Lg/Lr/Li). Apply the remap here explicitly since we loaded from Drive,
# not through ingestion._post_process which does this automatically.
df_work = df_raw.copy()
df_work["band"] = df_work["band"].replace(config.data.band_remap)
print(f"After remap:        {sorted(df_work['band'].dropna().unique())}")

# ── Quality cuts ──────────────────────────────────────────────────────────────
df = df_work[df_work["band"].isin(config.data.bands_use)].copy()
df = df[df["rmsmag"] <= config.data.rmsmag_max]
df = df[df["rmsmag"] > 0]
df = df.dropna(subset=["mag","rmsmag","mjd","band","provid"])
df = df.sort_values(["provid","mjd"]).reset_index(drop=True)

print(f"\nAfter quality cuts: {len(df):,} rows  |  {df['provid'].nunique():,} asteroids")
print(f"Bands kept: {sorted(df['band'].unique())}")
print(f"Date range: MJD {df['mjd'].min():.1f} — {df['mjd'].max():.1f}")
print(f"Sample provids: {sorted(df['provid'].unique())[:5]}")

# ── Quick summary (avoids pandas version issue with list_objects) ─────────────
summary = (df.groupby("provid")
             .agg(n_obs      =("mjd",  "count"),
                  baseline_d =("mjd",  lambda x: round(x.max()-x.min(), 1)),
                  mag_range  =("mag",  lambda x: round(x.max()-x.min(), 3)),
                  bands      =("band", lambda x: ",".join(sorted(x.unique()))))
             .reset_index()
             .sort_values("n_obs", ascending=False))
print(f"\nTop 20 by observation count:")
print(summary.head(20).to_string(index=False))


## Cell 5 — Ground truth from Greenstreet et al. 2026, Table 2

In [ ]:
import random

# All 76 known periods from Greenstreet et al. 2026.
# Key: provid as it appears in the MPC (space between year and designation).
# Period = LSM value (first of LSM/Fourier pair, hours).
# superfast = True if P <= 2.2hr from at least one method.
GROUND_TRUTH = {
    "2025 MA19":  {"period_hr": 8.9,   "amplitude": 0.7,  "superfast": False},
    "2025 MA45":  {"period_hr": 1.6,   "amplitude": 0.7,  "superfast": True},
    "2025 MA46":  {"period_hr": 5.9,   "amplitude": 0.6,  "superfast": False},
    "2025 MC34":  {"period_hr": 8.4,   "amplitude": 0.8,  "superfast": False},
    "2025 MD38":  {"period_hr": 15.8,  "amplitude": 1.1,  "superfast": False},
    "2025 MD40":  {"period_hr": 4.4,   "amplitude": 0.7,  "superfast": False},
    "2025 MD67":  {"period_hr": 7.8,   "amplitude": 1.2,  "superfast": False},
    "2025 MD76":  {"period_hr": 11.0,  "amplitude": 0.7,  "superfast": False},
    "2025 ME15":  {"period_hr": 6.9,   "amplitude": 0.9,  "superfast": False},
    "2025 ME24":  {"period_hr": 2.9,   "amplitude": 0.3,  "superfast": False},
    "2025 ME68":  {"period_hr": 0.9,   "amplitude": 0.6,  "superfast": True},
    "2025 MF76":  {"period_hr": 2.2,   "amplitude": 0.2,  "superfast": True},
    "2025 MG17":  {"period_hr": 4.3,   "amplitude": 0.4,  "superfast": False},
    "2025 MG56":  {"period_hr": 0.3,   "amplitude": 0.5,  "superfast": True},
    "2025 MH40":  {"period_hr": 8.0,   "amplitude": 1.4,  "superfast": False},
    "2025 MH69":  {"period_hr": 6.7,   "amplitude": 0.7,  "superfast": False},
    "2025 MH75":  {"period_hr": 4.2,   "amplitude": 0.5,  "superfast": False},
    "2025 MH86":  {"period_hr": 4.4,   "amplitude": 0.5,  "superfast": False},
    "2025 MJ13":  {"period_hr": 3.4,   "amplitude": 0.6,  "superfast": False},
    "2025 MJ21":  {"period_hr": 3.4,   "amplitude": 0.3,  "superfast": False},
    "2025 MJ23":  {"period_hr": 7.4,   "amplitude": 0.8,  "superfast": False},
    "2025 MJ30":  {"period_hr": 5.6,   "amplitude": 0.5,  "superfast": False},
    "2025 MJ71":  {"period_hr": 0.031, "amplitude": 0.4,  "superfast": True},
    "2025 MJ79":  {"period_hr": 1.0,   "amplitude": 0.2,  "superfast": True},
    "2025 MK23":  {"period_hr": 6.2,   "amplitude": 0.9,  "superfast": False},
    "2025 MK41":  {"period_hr": 0.063, "amplitude": 0.2,  "superfast": True},
    "2025 MK68":  {"period_hr": 5.0,   "amplitude": 0.7,  "superfast": False},
    "2025 MK83":  {"period_hr": 6.1,   "amplitude": 0.5,  "superfast": False},
    "2025 MK88":  {"period_hr": 2.7,   "amplitude": 0.4,  "superfast": False},
    "2025 ML10":  {"period_hr": 7.0,   "amplitude": 1.0,  "superfast": False},
    "2025 ML17":  {"period_hr": 6.7,   "amplitude": 0.6,  "superfast": False},
    "2025 ML35":  {"period_hr": 21.3,  "amplitude": 0.8,  "superfast": False},
    "2025 ML52":  {"period_hr": 11.5,  "amplitude": 0.7,  "superfast": False},
    "2025 ML53":  {"period_hr": 5.2,   "amplitude": 0.7,  "superfast": False},
    "2025 MM37":  {"period_hr": 3.7,   "amplitude": 0.3,  "superfast": True},
    "2025 MM81":  {"period_hr": 1.1,   "amplitude": 1.0,  "superfast": True},
    "2025 MM82":  {"period_hr": 5.0,   "amplitude": 0.8,  "superfast": False},
    "2025 MN25":  {"period_hr": 0.4,   "amplitude": 0.4,  "superfast": True},
    "2025 MN37":  {"period_hr": 4.8,   "amplitude": 0.8,  "superfast": False},
    "2025 MN45":  {"period_hr": 0.031, "amplitude": 0.4,  "superfast": True},
    "2025 MN7":   {"period_hr": 6.8,   "amplitude": 0.7,  "superfast": False},
    "2025 MO35":  {"period_hr": 6.3,   "amplitude": 0.5,  "superfast": False},
    "2025 MO39":  {"period_hr": 4.9,   "amplitude": 0.9,  "superfast": False},
    "2025 MO47":  {"period_hr": 9.1,   "amplitude": 0.7,  "superfast": False},
    "2025 MO79":  {"period_hr": 5.5,   "amplitude": 0.6,  "superfast": False},
    "2025 MP21":  {"period_hr": 6.2,   "amplitude": 0.6,  "superfast": False},
    "2025 MP47":  {"period_hr": 4.9,   "amplitude": 0.4,  "superfast": True},
    "2025 MP61":  {"period_hr": 3.0,   "amplitude": 0.6,  "superfast": False},
    "2025 MP67":  {"period_hr": 4.1,   "amplitude": 0.7,  "superfast": False},
    "2025 MP71":  {"period_hr": 9.1,   "amplitude": 0.4,  "superfast": False},
    "2025 MQ58":  {"period_hr": 2.9,   "amplitude": 0.3,  "superfast": False},
    "2025 MR33":  {"period_hr": 3.5,   "amplitude": 0.3,  "superfast": False},
    "2025 MS34":  {"period_hr": 2.3,   "amplitude": 0.5,  "superfast": False},
    "2025 MS7":   {"period_hr": 4.6,   "amplitude": 0.8,  "superfast": False},
    "2025 MT24":  {"period_hr": 8.9,   "amplitude": 1.0,  "superfast": False},
    "2025 MU10":  {"period_hr": 6.5,   "amplitude": 0.5,  "superfast": False},
    "2025 MU15":  {"period_hr": 0.4,   "amplitude": 0.5,  "superfast": True},
    "2025 MU24":  {"period_hr": 2.2,   "amplitude": 0.3,  "superfast": True},
    "2025 MU59":  {"period_hr": 8.2,   "amplitude": 0.6,  "superfast": False},
    "2025 MU8":   {"period_hr": 0.8,   "amplitude": 0.6,  "superfast": True},
    "2025 MU9":   {"period_hr": 4.9,   "amplitude": 0.6,  "superfast": False},
    "2025 MV19":  {"period_hr": 7.4,   "amplitude": 0.7,  "superfast": False},
    "2025 MV31":  {"period_hr": 5.2,   "amplitude": 0.7,  "superfast": False},
    "2025 MV38":  {"period_hr": 6.0,   "amplitude": 0.6,  "superfast": False},
    "2025 MV4":   {"period_hr": 5.9,   "amplitude": 0.8,  "superfast": False},
    "2025 MV46":  {"period_hr": 3.4,   "amplitude": 0.2,  "superfast": False},
    "2025 MV71":  {"period_hr": 0.2,   "amplitude": 0.4,  "superfast": True},
    "2025 MW70":  {"period_hr": 3.9,   "amplitude": 0.2,  "superfast": False},
    "2025 MX34":  {"period_hr": 5.8,   "amplitude": 0.7,  "superfast": False},
    "2025 MX44":  {"period_hr": 1.1,   "amplitude": 0.2,  "superfast": True},
    "2025 MX50":  {"period_hr": 1.9,   "amplitude": 0.3,  "superfast": True},
    "2025 MX63":  {"period_hr": 8.3,   "amplitude": 0.7,  "superfast": False},
    "2025 MX69":  {"period_hr": 9.1,   "amplitude": 0.7,  "superfast": False},
    "2025 MY23":  {"period_hr": 3.1,   "amplitude": 0.2,  "superfast": False},
    "2025 MY77":  {"period_hr": 7.6,   "amplitude": 1.0,  "superfast": False},
    "2025 MZ78":  {"period_hr": 1.2,   "amplitude": 0.5,  "superfast": True},
}

# ── Stratified 50/50 split ────────────────────────────────────────────────────
random.seed(42)
superfast = [p for p, v in GROUND_TRUTH.items() if v["superfast"]]
normal    = [p for p, v in GROUND_TRUTH.items() if not v["superfast"]]
random.shuffle(superfast); random.shuffle(normal)
validation_set = set(superfast[:len(superfast)//2] + normal[:len(normal)//2])
blind_set      = set(GROUND_TRUTH.keys()) - validation_set

print(f"Ground truth: {len(GROUND_TRUTH)} objects")
print(f"  Superfast (P<=2.2hr): {len(superfast)}")
print(f"  Normal:               {len(normal)}")
print(f"Validation set: {len(validation_set)}  |  Blind test set: {len(blind_set)}")

## Cell 6 — Check overlap with your dataset

In [ ]:
ground_truth_provids = set(GROUND_TRUTH.keys())
in_dataset = ground_truth_provids & set(df["provid"].unique())
missing    = ground_truth_provids - in_dataset

print(f"Greenstreet objects in dataset: {len(in_dataset)} / {len(GROUND_TRUTH)}")

if len(in_dataset) == 0:
    print("\n*** NO OVERLAP ***")
    print("Your CSV does not contain any RFL objects.")
    print("The RFL dataset requires obstime 2025-04-21 to 2025-05-06.")
    print("Sample provids in your dataset:", sorted(df['provid'].unique())[:5])
    print("\nRe-run the BigQuery query with the correct date range.")
else:
    print(f"Missing from dataset: {len(missing)}")
    if missing:
        print("  " + ", ".join(sorted(missing)[:10]))

    df_gt = df[df["provid"].isin(ground_truth_provids)].copy()
    print(f"\nRows for Greenstreet objects: {len(df_gt):,}")
    print(f"Observation counts:")
    obs_counts = df_gt.groupby("provid").size().sort_values(ascending=False)
    print(obs_counts.to_string())

## Cell 7 — Run pipeline on Greenstreet objects only

Expected runtime: ~10 minutes for 76 objects.

In [ ]:
from pipeline import run_pipeline

# Clear any old catalog so we start fresh
import os
old_cat = f"{RESULTS_DIR}/validation_catalog.csv"
if os.path.exists(old_cat):
    os.remove(old_cat)
    print("Cleared old catalog")

# Safety check
if 'df_gt' not in dir() or len(df_gt) == 0:
    raise RuntimeError(
        "df_gt is empty. Cell 6 must find overlap first.\n"
        "Check that your BQ query used obstime 2025-04-21 to 2025-05-06."
    )

print(f"Running pipeline on {df_gt['provid'].nunique()} asteroids...")
catalog = run_pipeline(df_gt, config=config, save_every_n=10)

print(f"\nCatalog: {len(catalog)} rows")
show_cols = ["provid","final_period_hr","reliability","r_code","r_flag",
             "t2_p_value","t2_mbls_fap","t2_mbls_band_support_frac"]
print(catalog[[c for c in show_cols if c in catalog.columns]].to_string(index=False))


## Cell 8 — Validation report

In [ ]:
import numpy as np

AGREE_TOL = 0.10

def compare_to_truth(pipe_p, truth_p, tol=AGREE_TOL):
    if np.isnan(pipe_p) or truth_p <= 0:
        return "no_period", np.nan
    delta      = abs(pipe_p - truth_p)      / truth_p
    delta_half = abs(pipe_p - truth_p/2.0)  / (truth_p/2.0)
    delta_dbl  = abs(pipe_p - truth_p*2.0)  / (truth_p*2.0)
    if delta      <= tol: return "exact",         round(delta*100, 1)
    if delta_half <= tol: return "half_period",   round(delta_half*100, 1)
    if delta_dbl  <= tol: return "double_period", round(delta_dbl*100, 1)
    return "disagree", round(delta*100, 1)

rows = []
for _, row in catalog.iterrows():
    provid = row["provid"]
    truth  = GROUND_TRUTH.get(provid)
    pipe_p = float(row.get("final_period_hr") or "nan")
    agree, delta = compare_to_truth(
        pipe_p, truth["period_hr"] if truth else np.nan
    )
    rows.append({
        "provid":       provid,
        "set":          "validation" if provid in validation_set else
                        ("blind" if provid in blind_set else "other"),
        "pipe_period":  pipe_p,
        "truth_period": truth["period_hr"] if truth else np.nan,
        "superfast":    truth["superfast"]  if truth else None,
        "agreement":    agree,
        "delta_pct":    delta,
        "r_code":       row.get("r_code"),
        "r_flag":       row.get("r_flag"),
        "reliability":  row.get("reliability"),
        "n_obs":        row.get("t1_n_obs"),
        "mhaov_p":      row.get("t2_p_value"),
        "mbls_fap":     row.get("t2_mbls_fap"),
        "band_frac":    row.get("t2_mbls_band_support_frac"),
    })

val = pd.DataFrame(rows)

def print_section(title, df):
    if len(df) == 0: return
    total     = len(df)
    pub       = int(df["r_code"].ge(1).sum())
    has_truth = df[df["truth_period"].notna()]
    exact     = int((has_truth["agreement"]=="exact").sum())
    half      = int((has_truth["agreement"]=="half_period").sum())
    dbl       = int((has_truth["agreement"]=="double_period").sum())
    disagree  = int((has_truth["agreement"]=="disagree").sum())
    no_p      = int((has_truth["agreement"]=="no_period").sum())

    print(f"\n{'='*70}")
    print(title)
    print(f"{'='*70}")
    print(f"Objects:           {total}")
    print(f"T1 passed:         {int(df['r_code'].ge(0).sum() - df['r_code'].eq(0).sum() + df['reliability'].eq('t1_rejected').sum())}")
    print(f"Published (R>=1):  {pub}  ({pub/total*100:.0f}%)")
    print(f"  R=3: {int(df['r_code'].eq(3).sum())}  R=2: {int(df['r_code'].eq(2).sum())}  "
          f"R=1: {int(df['r_code'].eq(1).sum())}  R=-1: {int(df['r_code'].eq(-1).sum())}  "
          f"R=0: {int(df['r_code'].eq(0).sum())}")

    if len(has_truth):
        print(f"\nPeriod recovery ({len(has_truth)} with known period):")
        print(f"  exact (<10%):      {exact}  ({exact/len(has_truth)*100:.0f}%)")
        print(f"  half_period (P/2): {half}")
        print(f"  double_period (2P):{dbl}")
        print(f"  disagree:          {disagree}")
        print(f"  no period found:   {no_p}")
        sf = has_truth[has_truth["superfast"]==True]
        nm = has_truth[has_truth["superfast"]==False]
        if len(sf):
            sf_ok = int((sf["agreement"].isin(["exact","half_period","double_period"])).sum())
            print(f"\n  Superfast (P<=2.2hr): {sf_ok}/{len(sf)} recovered")
        if len(nm):
            nm_ok = int((nm["agreement"]=="exact").sum())
            print(f"  Normal speed:         {nm_ok}/{len(nm)} exact")

    print(f"\nDetailed:")
    print(df[["provid","pipe_period","truth_period","agreement",
              "delta_pct","r_code","r_flag"]].to_string(index=False))

val_df   = val[val["set"]=="validation"]
blind_df = val[val["set"]=="blind"]
other_df = val[val["set"]=="other"]

print_section("VALIDATION SET (known to us during development)", val_df)
print_section("BLIND TEST SET (held out)", blind_df)
if len(other_df):
    pub_o = int(other_df["r_code"].ge(1).sum())
    print(f"\nOther objects (not in Greenstreet table): {len(other_df)}, published: {pub_o}")

## Cell 9 — Phase-folded lightcurve plots

In [ ]:
import matplotlib.pyplot as plt

BAND_COLORS = {"Lg":"#2196F3","Lr":"#F44336","Li":"#FF9800","Lu":"#9C27B0"}
R_COLORS    = {3:"#2e7d32", 2:"#1565c0", 1:"#e65100", 0:"#b71c1c", -1:"#6a1b9a"}

def plot_folded(ax, df_obj, period_hr, title, r_code=None, truth_p=None):
    t0    = df_obj["mjd"].min()
    t     = (df_obj["mjd"].values - t0) * 24.0
    phase = (t % period_hr) / period_hr
    for band in sorted(df_obj["band"].unique()):
        m = df_obj["band"]==band
        c = BAND_COLORS.get(band,"gray")
        ax.errorbar(phase[m],   df_obj["mag"].values[m], yerr=df_obj["rmsmag"].values[m],
                    fmt="o", ms=3, alpha=0.8, color=c, label=band, capsize=0)
        ax.errorbar(phase[m]+1, df_obj["mag"].values[m], yerr=df_obj["rmsmag"].values[m],
                    fmt="o", ms=3, alpha=0.25, color=c, capsize=0)
    ax.invert_yaxis()
    ax.set_xlabel("Phase"); ax.set_ylabel("Mag")
    lbl = title
    if truth_p: lbl += f"\n[truth={truth_p:.3f}hr]"
    ax.set_title(f"{lbl}\nP={period_hr:.3f}hr  R={r_code}",
                 color=R_COLORS.get(r_code,"k"), fontsize=8)
    ax.legend(fontsize=7); ax.set_xlim(0,2)

targets = (val[val["r_code"].ge(1) & val["pipe_period"].notna()]
           .sort_values("r_code", ascending=False).head(12))

if len(targets) == 0:
    print("No published periods to plot.")
else:
    ncols = 3
    nrows = int(np.ceil(len(targets)/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
    axes = np.array(axes).flatten()
    for i, (_, row) in enumerate(targets.iterrows()):
        df_obj  = df_gt[df_gt["provid"]==row["provid"]].copy()
        truth   = GROUND_TRUTH.get(row["provid"])
        truth_p = truth["period_hr"] if truth else None
        plot_folded(axes[i], df_obj, row["pipe_period"],
                    row["provid"], int(row["r_code"]), truth_p)
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle("Phase-folded lightcurves — top results by R-code", fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/folded_lightcurves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {RESULTS_DIR}/folded_lightcurves.png")


## Cell 10 — Deep dive: single asteroid diagnostics

Change `INSPECT_PROVID` to any asteroid in the dataset.

In [ ]:
INSPECT_PROVID = "2025 MK41"   # <-- change to any provid

from ingestion import load_single_object
from preprocessing import preprocess
from tier1 import run_tier1
from tier2 import run_tier2
from characterise import characterise

df_obj = load_single_object(INSPECT_PROVID, df_gt)
data   = preprocess(df_obj, config)
char   = characterise(df_obj)
t1     = run_tier1(data, config)

print(f"{'='*60}\n{INSPECT_PROVID}\n{'='*60}")
print(f"N={data.n_obs}  bands={data.band_counts}  baseline={data.baseline_hr:.1f}hr")
print(f"Period floor: {data.period_min_hr*60:.2f} min  (Eyer & Bartholdi 1999)")
print(f"SNR={data.snr:.2f}  regime={char.regime}  ceiling={char.reliability_ceiling}")
print(f"T1: passes={t1.passes}  GLS={t1.best_period_gls:.4f}hr  MBLS={t1.best_period_mbls:.4f}hr")
print(f"    gls_cont={t1.gls_contamination:.2f}  mbls_cont={t1.mbls_contamination:.2f}")

truth = GROUND_TRUTH.get(INSPECT_PROVID)
if truth:
    print(f"Truth: P={truth['period_hr']}hr  superfast={truth['superfast']}")

t2 = None
if t1.passes:
    t2 = run_tier2(data, t1, config)
    print(f"T2: MHAOV={t2.best_period_mhaov:.4f}hr  MBLS={t2.best_period_mbls:.4f}hr  CE={t2.best_period_ce:.4f}hr")
    print(f"    agreement={t2.agreement}  consensus={t2.consensus_period:.4f}hr")
    print(f"    MHAOV p={t2.p_value:.2e} ({'sig' if t2.mhaov_sig else 'not sig'})  "
          f"MBLS FAP={t2.mbls_fap:.4f} ({'sig' if t2.mbls_sig else 'not sig'})  both={t2.both_sig}")
    print(f"    band_support={t2.mbls_band_support}")
    print(f"    frac={t2.mbls_band_support_frac:.2f}  n_bands={t2.mbls_n_bands_supporting}")

# 4-panel figure
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f"{INSPECT_PROVID} — Diagnostics", fontsize=12)

ax = axes[0,0]
for band in sorted(df_obj["band"].unique()):
    m = df_obj["band"]==band
    ax.errorbar(df_obj["mjd"].values[m], df_obj["mag"].values[m],
                yerr=df_obj["rmsmag"].values[m], fmt="o", ms=3, alpha=0.7,
                label=band, color=BAND_COLORS.get(band,"gray"))
ax.invert_yaxis(); ax.set_xlabel("MJD"); ax.set_ylabel("Mag")
ax.set_title("Raw lightcurve"); ax.legend(fontsize=8)

ax = axes[0,1]
if t1.passes and len(t1.test_periods)>0:
    ax.plot(t1.test_periods, t1.gls_power, "b-", lw=0.8, alpha=0.8, label="GLS")
    wp_n = t1.window_power/(t1.window_power.max()+1e-12)*t1.gls_power.max()
    ax.fill_between(t1.test_periods, wp_n, alpha=0.2, color="orange", label="Window")
    ax.axvline(t1.best_period_gls, color="b", lw=1.5, ls="--",
               label=f"best={t1.best_period_gls:.3f}hr")
    if truth:
        ax.axvline(truth["period_hr"], color="g", lw=1.5, ls=":",
                   label=f"truth={truth['period_hr']}hr")
ax.set_xlabel("Period (hr)"); ax.set_ylabel("GLS power")
ax.set_title(f"T1 GLS (cont={t1.gls_contamination:.2f})"); ax.legend(fontsize=8)

ax = axes[1,0]
if t2:
    mh_n = t2.mhaov_power/(t2.mhaov_power.max()+1e-12)
    mb_n = t2.mbls_power /(t2.mbls_power.max() +1e-12)
    ax.plot(t2.test_periods, mh_n, "g-", lw=0.8, alpha=0.8, label="MHAOV (norm)")
    ax.plot(t2.test_periods, mb_n, "r-", lw=0.8, alpha=0.8, label="MBLS (norm)")
    ax.axvline(t2.best_period_mhaov, color="g", lw=1.5, ls="--",
               label=f"MHAOV={t2.best_period_mhaov:.3f}hr")
    ax.axvline(t2.best_period_mbls,  color="r", lw=1.5, ls=":",
               label=f"MBLS={t2.best_period_mbls:.3f}hr")
    if truth:
        ax.axvline(truth["period_hr"], color="k", lw=1.5, ls=":",
                   label=f"truth={truth['period_hr']}hr")
ax.set_xlabel("Period (hr)"); ax.set_ylabel("Norm power")
ax.set_title(f"T2 MHAOV+MBLS  agree={t2.agreement if t2 else 'N/A'}"); ax.legend(fontsize=8)

ax = axes[1,1]
pipe_row = val[val["provid"]==INSPECT_PROVID]
if len(pipe_row) and not np.isnan(float(pipe_row.iloc[0]["pipe_period"] or "nan")):
    p  = float(pipe_row.iloc[0]["pipe_period"])
    rc = int(pipe_row.iloc[0]["r_code"])
    plot_folded(ax, df_obj, p, INSPECT_PROVID, rc,
                truth["period_hr"] if truth else None)
else:
    ax.text(0.5,0.5,"No period published",ha="center",va="center",transform=ax.transAxes)
    ax.set_title(f"{INSPECT_PROVID} — no period")

plt.tight_layout()
fname = f"{RESULTS_DIR}/{INSPECT_PROVID.replace(' ','_')}_diagnostics.png"
plt.savefig(fname, dpi=150, bbox_inches="tight"); plt.show()
print(f"Saved: {fname}")


## Cell 11 — Save and download results

In [ ]:
val.to_csv(f"{RESULTS_DIR}/validation_report.csv", index=False)
catalog.to_csv(f"{RESULTS_DIR}/pipeline_catalog.csv", index=False)
print("Saved to Drive:")
print(f"  {RESULTS_DIR}/validation_report.csv")
print(f"  {RESULTS_DIR}/pipeline_catalog.csv")

print(f"\nDrive path: MyDrive/{DRIVE_FOLDER_NAME}/pipeline_results/")

from google.colab import files
files.download(f"{RESULTS_DIR}/validation_report.csv")
files.download(f"{RESULTS_DIR}/pipeline_catalog.csv")
